# Interactive Cosmograph visualization of SSNs for MGnify proteins

TODO: talk bla bla, mgnfiy logo

## Setting up the environment

### Install required packages

In [1]:
%pip install duckdb cosmograph pyarrow ipywidgets

Note: you may need to restart the kernel to use updated packages.


### Import libraries

In [2]:
import pandas as pd
from cosmograph import cosmo
import ipywidgets as widgets_ipy
from IPython.display import display
from cosmograph import cosmo
import duckdb

### Define cluster representant

In [15]:
CLUSTER_REP="MGYP008633949747"

## 1. Load data
Choose one of the options below

### Option A: Read files using remote FTP

In [70]:
base_url = f"https://ftp.ebi.ac.uk/pub/databases/metagenomics/gsoc_ssn/networks/{CLUSTER_REP}/"

nodes_url = f"{base_url}{CLUSTER_REP}_nodes.parquet"
edges_url = f"{base_url}{CLUSTER_REP}_edges.parquet"

nodes_df = duckdb.sql(f"SELECT * FROM '{nodes_url}'").df()
edges_df = duckdb.sql(f"SELECT * FROM '{edges_url}'").df()

display(nodes_df.head())
display(edges_df.head())

,id,biome
0,MGYP002855862099,root:Host-associated:Mammals:Digestive system:...
1,MGYP002673540755,root:Host-associated:Human:Digestive system:La...
2,MGYP002853144258,root:Host-associated:Mammals:Digestive system:...
3,MGYP002860676619,root:Host-associated:Mammals:Digestive system:...
4,MGYP002627048376,root:Host-associated:Mammals:Digestive system:...


,source,target,sequence_identity
0,MGYP002656457721,MGYP002774926903,98.8
1,MGYP002774926903,MGYP008339877134,98.6
2,MGYP000822534626,MGYP002774926903,98.0
3,MGYP002774926903,MGYP008603557989,98.2
4,MGYP002774926903,MGYP004472909437,98.4


### Option B: Read files locally

In [9]:
nodes_df = pd.read_parquet("MGYP000134696410_nodes.parquet")
edges_df = pd.read_parquet("MGYP000134696410_edges.parquet")

# Quick view of data structure
display(nodes_df.head(5))
display(edges_df.head(5))

,id,biome
0,MGYP000102271628,root:Host-associated:Human:Digestive system:La...
1,MGYP000606320263,root:Host-associated:Human:Digestive system
2,MGYP001470561486,root:Host-associated:Human:Digestive system:La...
3,MGYP007403529906,root:Host-associated:Human:Digestive system:Hi...
4,MGYP009635841147,root:Host-associated:Human:Digestive system:La...


,source,target,sequence_identity
0,MGYP004547913711,MGYP008685658098,99.6
1,MGYP000699390189,MGYP004547913711,100.0
2,MGYP002703219698,MGYP004547913711,100.0
3,MGYP004547913711,MGYP009013785984,100.0
4,MGYP004547913711,MGYP005970380175,100.0


## 2. Create Cosmograph blueprint

Function to set the cluster representant's size and shape different from the other points.

In [74]:
def set_cluster_rep_style(kwargs):
    # Make sure 'points' exist in the kwargs
    if "points" in kwargs and kwargs["points"] is not None:
        points_df = kwargs["points"]
        
        # Add the size and shape column to the temporary copy
        points_df['size'] = 5.0
        points_df.loc[points_df['id'] == CLUSTER_REP, 'node'] = 75.0
        
        # Tell the widget to use this new temporary column
        kwargs["point_size_by"] = "size"
        
    return kwargs

We create a reusable template, determining the style.
All configuration parameters can be found here [here](https://cosmograph.app/docs-widget/configuration/).

In [123]:
graph_style = cosmo(
    point_id_by="id",
    link_source_by="source",
    link_target_by="target",
    ingress=set_cluster_rep_style,
    
    point_color_by="biome",
    #TODO: figure out coloring strategy here
    #figure out what is issue with size 
    point_size_strategy="direct",
    point_label_by="id",
    fit_view_on_init=False,
    select_point_on_click=True,
    select_point_on_label_click=True,
    reset_selection_on_empty_canvas_click=True,
    point_greyout_opacity=0.2, 
    link_greyout_opacity=0.2,
    render_hovered_point_ring=True,
    enable_drag=True,
    show_labels_for=[CLUSTER_REP],
    disable_point_size_legend=True,
)
#TODO: set the params here

# Calling the widget object
This renders the graph locally using your GPU.

In [124]:
graph = graph_style(points=nodes_df, links=edges_df)
graph